## 样例：flash_attention 算子集成至 PyTorch

一个示例项目，把 Tilelang-Ascend 的 [flash_attention](../flash_attention/flash_attn_bhsd.py) 算子以 Python C 扩展的形式集成至 PyTorch

In [1]:
!pwd  # 项目路径

/mnt/workspace/tilelang-ascend/examples/torch_tl_ascend


```
torch_tl_ascend/
├── compile_tl_op        # 用于编译 Tilelang-Ascend 算子为 .so 文件的工具脚本
├── src
│   ├── torch_tl_ascend  # torch_tl_ascend 包
│   └── _inner.cpp       # Python C 模块，包装算子调用，注册至 PyTorch
├── setup.py             # 安装脚本
├── test_torch.py        # 测试脚本
└── ...
```

具体来说，它会提供一个叫 `torch-tl-ascend` 的 Python 包，让我们先安装它

In [2]:
!python setup.py install

Grabbing .py of <function flash_attention_fwd at 0xfffebd01f6a0> (/mnt/workspace/tilelang-ascend/examples/flash_attention/flash_attn_bhsd.py)
Grabbing .so of <function flash_attention_fwd at 0xfffebd01f6a0> with B=4, S=4096, H=16, D=128
Got libop.so
running install
/mnt/workspace/tilelang-ascend/venv/lib/python3.11/site-packages/setuptools/command/install.py:34: SetuptoolsDeprecationWarning: setup.py install is deprecated. Use build and pip and other standards-based tools.
  warnings.warn(
/mnt/workspace/tilelang-ascend/venv/lib/python3.11/site-packages/setuptools/command/easy_install.py:144: EasyInstallDeprecationWarning: easy_install command is deprecated. Use build and pip and other standards-based tools.
  warnings.warn(
running bdist_egg
running egg_info
writing /mnt/workspace/tilelang-ascend/examples/torch_tl_ascend/src/torch_tl_ascend.egg-info/PKG-INFO
writing dependency_links to /mnt/workspace/tilelang-ascend/examples/torch_tl_ascend/src/torch_tl_ascend.egg-info/dependency_link

In [3]:
# 查看安装好的 torch-tl-ascend 包
!pip show torch-tl-ascend

Name: torch-tl-ascend
Version: 0.1.0
Summary: Torch TL Ascend
Home-page: 
Author: 
Author-email: 
License: 
Location: /mnt/workspace/tilelang-ascend/venv/lib/python3.11/site-packages/torch_tl_ascend-0.1.0-py3.11-linux-aarch64.egg
Requires: 
Required-by: 


来测试一下（逻辑与测试脚本 [./test_torch.py](./test_torch.py) 一致）：

In [4]:
import torch
import torch_tl_ascend  # 导入时，会将算子注册到 torch.ops.tl_ascend 下

def ref_flash_attention(q, k, v):
    q = q.float()
    k = k.float()
    v = v.float()

    acc = torch.einsum("bhsd,bhkd->bhsk", q, k) * (1.0 / q.shape[-1])**0.5
    acc = acc.softmax(dim=-1)
    o = torch.einsum("bhsk,bhkd->bhsd", acc, v)
    return o.to(torch.float16)


B, S, H, D = 4, 4096, 16, 128

torch.set_default_device('npu')
torch.manual_seed(0)

q = torch.randn((B, H, S, D), dtype=torch.float16)
k = torch.randn((B, H, S, D), dtype=torch.float16)
v = torch.randn((B, H, S, D), dtype=torch.float16)

torch.npu.synchronize()
print("Init Successful!")


/mnt/workspace/tilelang-ascend/venv/lib/python3.11/site-packages/torch_npu/__init__.py:309: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default.                      Do not set it to 1 to prevent some unknown errors
  warnings.warn("On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. \


Init Successful!


In [5]:
# 调用注册好的 torch.ops.tl_ascend.flash_attention
output = torch.ops.tl_ascend.flash_attention(q, k, v)

ref_output = ref_flash_attention(q, k, v)
torch.npu.synchronize()

torch.testing.assert_close(ref_output, output, rtol=1e-2, atol=1e-2)

print("Test Passed!")

Test Passed!


### 原理解析

在 `torch-tl-ascend` 包安装时，工具脚本 [./compile_tl_op/flash_attention.py](./compile_tl_op/flash_attention.py) 会被调用

类似 JIT 模式，大致经历了如下的过程：

---
```python
# 获取被 @tilelang.jit 装饰过的算子函数（flash_attention_fwd）
op_func = get_jit_op_func()

# 按给定的输入编译出 kernel 函数
B, S, H, D = 4, 4096, 16, 128
print("Grabbing .so of", op_func, f"with B={B}, S={S}, H={H}, D={D}")
kernel = get_kernel(op_func, B, S, H, D)

# 获取编译出的 .so 文件的路径
lib_so_path = so_path_of(kernel)
package_so_path = PACKAGE_ROOT / SO_NAME

# 复制 .so 到 torch-tl-ascend 包的目录中（src/torch_tl_ascend）
shutil.copy(lib_so_path, package_so_path)
print("Got", package_so_path.name)
```
---
由此，使得打包出的 `torch-tl-ascend` 包含编译好的算子 .so

`torch-tl-ascend` 包里的算子 .so 会被链接到 [./src/_inner.cpp](./src/_inner.cpp) 中，大致流程如下：

定义 Torch 层包装函数，输入 `at::Tensor`，调用算子 `call` 接口（类似 AOT 模式中通过 .so 调用算子的流程）

```cpp
at::Tensor flash_attention_wrapper(const at::Tensor& Q, const at::Tensor& K, const at::Tensor& V) {
    // ...

    call(
        reinterpret_cast<uint8_t*>(Q.data_ptr()), 
        reinterpret_cast<uint8_t*>(K.data_ptr()), 
        reinterpret_cast<uint8_t*>(V.data_ptr()), 
        reinterpret_cast<uint8_t*>(Output.data_ptr()),
        reinterpret_cast<uint8_t*>(workspace_1.data_ptr()),
        reinterpret_cast<uint8_t*>(workspace_2.data_ptr()),
        reinterpret_cast<uint8_t*>(workspace_3.data_ptr()),
        stream
    );
    return Output;
}
```

注册包装函数到 `torch` 的 `ops.tl_ascend` 命名空间下

```cpp
TORCH_LIBRARY(tl_ascend, m) {
    m.def("flash_attention(Tensor Q, Tensor K, Tensor V) -> Tensor");
}

TORCH_LIBRARY_IMPL(tl_ascend, PrivateUse1, m) {  // 注册 NPU 通路用 PrivateUse1
    m.impl("flash_attention", &flash_attention_wrapper);
}
```

定义一个空的 Python C 模块，当此模块被从 Python 侧导入时，会触发上面注册包装函数的逻辑
```cpp
extern "C" {
    PyObject* PyInit__inner(void)
    {
        static struct PyModuleDef module_def = {
            PyModuleDef_HEAD_INIT,
            "_inner",   /* 模块名 */
            NULL,       /* 描述文档 */
            -1,         /* 状态大小 */
            NULL,       /* 方法列表 */
        };
        return PyModule_Create(&module_def);
    }
}
```

[torch-tl-ascend 包](./src/torch_tl_ascend/__init__.py) 加载时，会自动导入上述 Python C 模块

```python
import torch_tl_ascend._inner
```

由此，实现了将算子集成至 PyTorch

---

### 扩展样例：flash_attention 算子集成至 LibTorch

[demo_libtorch](./demo_libtorch) 子目录提供了另一个 C++ 版本的样例，把 [flash_attention](../flash_attention/flash_attn_bhsd.py) 算子集成至 LibTorch（PyTorch 的 C++ 接口）

作为扩展，这里也简单介绍一下

```
demo_libtorch/
├── build.sh             # 编译脚本
├── prepare.py           # 准备算子 .so 文件的脚本（基于 ../compile_tl_op 中的工具脚本）
├── CMakeLists.txt       # CMake 配置
├── flash_attention.cpp  # 主程序
└── lib/                 # 存放算子 .so 文件的目录
```

编译时，[build.sh](./demo_libtorch/build.sh) 会调用 [prepare.py](./demo_libtorch/prepare.py)

仍然借助工具脚本 [../compile_tl_op/flash_attention.py](./compile_tl_op/flash_attention.py) 获取算子的 .so 文件

主程序 [flash_attention.cpp](./demo_libtorch/flash_attention.cpp) 中

依旧定义 Torch 层包装函数

```cpp
at::Tensor flash_attention_fwd(const at::Tensor& Q, const at::Tensor& K, const at::Tensor& V) {
    ...
}
```

在 C++ 层面调用包装函数时，使用 [libtorch_npu](https://www.hiascend.com/document/detail/zh/Pytorch/730/configandinstg/instg/docs/zh/installation_guide/building_libtorch_npu.md) 配合 LibTorch 创建 NPU 上的张量

可以编写出类似 Python 版本的测试逻辑：

```cpp
int main() {
    constexpr int64_t B = 4;
    constexpr int64_t S = 4096;
    constexpr int64_t H = 16;
    constexpr int64_t D = 128;

    torch_npu::init_npu("npu:0");
    auto device = torch::Device("npu:0");

    torch::manual_seed(0);

    auto options = torch::TensorOptions().dtype(at::kHalf).device(device);
    torch::Tensor q = torch::randn({B, H, S, D}, options);
    torch::Tensor k = torch::randn({B, H, S, D}, options);
    torch::Tensor v = torch::randn({B, H, S, D}, options);

    torch::npu::synchronize();
    std::cout << "init successful!" << std::endl;

    torch::Tensor output = flash_attention_fwd(q, k, v);
    torch::Tensor ref_output = ref_flash_attn(q, k, v);

    torch::npu::synchronize();

    TORCH_CHECK(
        torch::allclose(ref_output, output, /*rtol=*/ 1e-2, /*atol=*/ 1e-2),
        "Flash Attention output does NOT match reference!"
    );

    torch_npu::finalize_npu();

    std::cout << "Test Passed!" << std::endl;
    return 0;
}
```